If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec11_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 11: Pivots and Joins
v.ekc

Two power tools today: **pivot** turns two categorical columns into a grid of summaries, and **join** glues two tables together on a shared column. With these, `group` gets serious company. (Slides: KL Lecture 11 deck.)

**Today's sections:**
1. Rows from lists
2. Review — grouping
3. Cross-classification
4. Pivot tables
5. Joins
6. Try It — Skyscrapers

In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.filterwarnings("ignore")

---
## 1. Rows from Lists

So far we've built tables column by column. You can also start from the labels and add **rows** (as lists — remember, lists can mix types).

In [ ]:
Table().with_columns('Numbers', np.arange(1, 4))

In [ ]:
drinks = Table(['Drink', 'Cafe', 'Price'])
drinks

In [ ]:
drinks = drinks.with_rows([
    ['Matcha Latte', 'Northtown', 5.5],
    ['Espresso', 'Cafe Mokka',  2.75],
    ['Latte',    'Cafe Mokka',  5.25],
    ['Espresso', "Jitter Bean",   2]
])
drinks

---
## 2. Review — Grouping

Warm up on the welcome survey: `group` with one column counts; with a function, it summarizes.

In [ ]:
survey = Table.read_table('data111_survey_fa24.csv')
survey.show(3)

In [ ]:
survey.group("Handedness")

In [ ]:
(survey
 .select('Handedness', 'Hours of Sleep')
 .group("Handedness", np.average))

---
## 3. Cross-Classification

Group by a *list* of two columns → one row per combination.

In [ ]:
survey.group(["Handedness", 'Sleep Position'])

In [ ]:
(survey
 .select("Handedness", 'Sleep Position', "Hours of Sleep")
 .group(["Handedness", 'Sleep Position'], collect=np.average))

---
## 4. Pivot Tables

A pivot shows the same information as a two-column group — but as a **grid**: one variable across the top, one down the side. Much easier to compare. (KL deck, slides 14–15: group or pivot? If you want a grid, pivot.)

### 📋 Board Reference

| Code | What it does |
|---|---|
| `t.pivot('cols', 'rows')` | grid of **counts** |
| `t.pivot('cols', 'rows', values='v', collect=f)` | grid of `f` applied to `v` |
| gotcha | `values` and `collect` come **together** or not at all |

In [ ]:
survey.pivot('Sleep Position', 'Handedness')

In [ ]:
survey.pivot('Sleep Position', 
             'Handedness', 
             values='Hours of Sleep', 
             collect=np.average)

In [ ]:
# This cell will throw an error
# You cannot include just 1 of the optional arguments
survey.pivot('Sleep Position', 
             'Handedness', 
             collect=np.average)

---
## 5. Joins

When two tables share a column, `join` matches their rows up. (KL deck, slides 17–19.)

### 📋 Board Reference

| Code | What it does |
|---|---|
| `a.join('col_a', b, 'col_b')` | keep rows where the values match |
| unmatched rows | quietly dropped! |
| repeated values | every matching pair appears |

In [ ]:
drinks

In [ ]:
discounts = Table().with_columns(
    'Coupon % off', make_array(10, 25, 5),
    'Location', make_array('Northtown', 'Cafe Mokka', 'Northtown')
)
discounts

In [ ]:
combined = drinks.join('Cafe', discounts, 'Location')
combined

In [ ]:
discounted_frac = 1 - combined.column('Coupon % off') / 100
combined.with_column(
    'Discounted Price', 
    combined.column('Price') * discounted_frac
)

In [ ]:
drinks.join('Cafe', drinks, 'Cafe')

> **Look closely at that last one** — joining `drinks` with *itself* pairs every drink with every other drink at the same cafe. Joins multiply matching rows; always sanity-check `num_rows` after a join.

---
### Try It 1: Skyscrapers 🏙️

Run the next cell to load ~1,700 U.S. skyscrapers (name, material, city, height, year completed).

1. For each city, how tall is the tallest building for each material?

2. For each city, what's the height difference between the tallest steel building and the tallest concrete building?

3. Generate a table of the **names** of the oldest buildings for each material for each city. *Hint: sort first — then what does `pivot` collect?*

In [ ]:
# From the CORGIS Dataset Project
# By Austin Cory Bart acbart@vt.edu
# Version 2.0.0, created 3/22/2016
# https://corgis-edu.github.io/corgis/csv/skyscrapers/

sky = Table.read_table('skyscrapers.csv')
sky = (sky.with_column('age', 2024 - sky.column('completed'))
          .drop('completed'))
sky.show(3)

In [ ]:
# 1. For each city, how tall is the tallest building for each material?



In [ ]:
# 2. For each city, what’s the height difference between the tallest 
#    steel building and the tallest concrete building?









In [ ]:
# 3. Generate a table of the names of the oldest buildings for each 
#    material for each city:

# Hint: You can use sort to find the name of the oldest building in the dataset
sky.sort('age', descending=True).column('name').item(0)


# Put your solution here








<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*All three are pivots — the only choices are `values` and `collect`.*

```python
# 1.
tallest = sky.pivot('material', 'city', values='height', collect=max)
tallest

# 2.
tallest.with_column(
    'difference',
    abs(tallest.column('steel') - tallest.column('concrete')))

# 3.
def first(s):
    return s.item(0)

sky.sort('completed').pivot('material', 'city', values='name', collect=first)
```

</details>

---
> **Today's takeaway:** group counts, pivot grids, join glues. We'll practice all three on a big real dataset next lecture.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — pivot & join
```python
t.pivot('col_across', 'col_down', values='v', collect=np.average)
a.join('shared_col', b, 'its_name_in_b')
```